# Precipitation Calcs
This notebook calculates the daily spatial mean for each watershed in the WUS focus, regardless of whether or not there was an AR that day or not. However, only values > 1 mm are kept in averaging.

In [1]:
import xarray as xa
import geopandas as gpd
import pandas as pd
import numpy as np
import regionmask
from glob import glob

ERROR 1: PROJ: proj_create_from_database: Open of /glade/u/home/tcorrie/.conda/envs/rmask/share/proj failed


In [2]:
wusd3 = pd.read_csv('../wusd3_gcms.csv', index_col = 0)
for i in range(1011, 1201, 20):
    wusd3.loc[len(wusd3)] = ['cesm2-le', 'CESM2-LE', str(i), '365_day']


wusd3_cmip6nonbc = wusd3.iloc[[0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15]]
wusd3_cmip6bc = wusd3.iloc[[0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15]]
wusd3_cesm2le = wusd3.iloc[16:]
wusd3_wrfera5 = wusd3.iloc[[6]]
wusd3_prism = wusd3.iloc[[6]]

for row in wusd3_cmip6nonbc.index:
    print(wusd3_cmip6nonbc.loc[row, 'Model'])

access-cm2
canesm5
cesm2
cnrm-esm2-1
ec-earth3
ec-earth3-veg
fgoals-g3
giss-e2-1-g
miroc6
mpi-esm1-2-hr
mpi-esm1-2-hr
mpi-esm1-2-lr
noresm2-mm
taiesm1
ukesm1-0-ll


In [12]:
# Non-BC CMIP6 ensemble

regions = gpd.read_file('/glade/u/home/tcorrie/watersheds_WUS.geojson', driver='GeoJSON')

for row in wusd3_cmip6nonbc.index:
    #if row <= 6:
    #    continue
    model = wusd3_cmip6nonbc.loc[row, 'Model']
    member = wusd3_cmip6nonbc.loc[row, 'Member']
    print(f"{model}/{member}")

    glob_files = glob(f'/glade/derecho/scratch/tcorrie/WRFGCM/prec/prec.daily.{model}.{member}.*.glob.nc')
    glob_files = [gf for gf in glob_files if "_bc." not in gf]
    prec_ds = xa.open_mfdataset(glob_files, use_cftime=True).load()
    prec_rmask = regionmask.mask_geopandas(regions, prec_ds.XLONG, prec_ds.XLAT)
    
    prec_regional = xa.Dataset(
        coords = dict(time=("time", prec_ds.day.values), region=('region', regions.huc4.values)),
        data_vars = dict(prec_mean=(['time', 'region'], np.empty((len(prec_ds.day.values), len(regions.huc4.values)))))
    )
    
    for idx, reg in enumerate(regions.huc4.values): # Loop through each watershed (24 total)
        prec_wshed = prec_ds.prec.where(prec_rmask == idx).where(prec_ds.prec >= 1).mean(dim=['lat2d','lon2d'], skipna=True)
        prec_regional.prec_mean.loc[{'region': reg}] = prec_wshed.values

    prec_regional.to_netcdf(f'/glade/work/tcorrie/ARdata/pmean/prec_mean.cmip6nonbc.{model}.{member}.nc')

fgoals-g3/r1i1p1f1
giss-e2-1-g/r1i1p1f2
miroc6/r1i1p1f1
mpi-esm1-2-hr/r3i1p1f1
mpi-esm1-2-hr/r7i1p1f1
mpi-esm1-2-lr/r7i1p1f1
noresm2-mm/r1i1p1f1
taiesm1/r1i1p1f1
ukesm1-0-ll/r2i1p1f2


In [13]:
# BC CMIP6 ENSEMBLE

regions = gpd.read_file('/glade/u/home/tcorrie/watersheds_WUS.geojson', driver='GeoJSON')

for row in wusd3_cmip6bc.index:
    model = wusd3_cmip6bc.loc[row, 'Model']
    member = wusd3_cmip6bc.loc[row, 'Member']
    print(f"{model}/{member}")

    glob_files = glob(f'/glade/derecho/scratch/tcorrie/WRFGCM/prec/prec.daily.{model}.{member}.*.glob.nc')
    glob_files = [gf for gf in glob_files if "_bc." in gf]
    prec_ds = xa.open_mfdataset(glob_files, use_cftime=True).load()
    prec_rmask = regionmask.mask_geopandas(regions, prec_ds.XLONG, prec_ds.XLAT)
    
    prec_regional = xa.Dataset(
        coords = dict(time=("time", prec_ds.day.values), region=('region', regions.huc4.values)),
        data_vars = dict(prec_mean=(['time', 'region'], np.empty((len(prec_ds.day.values), len(regions.huc4.values)))))
    )
    
    for idx, reg in enumerate(regions.huc4.values):
        prec_wshed = prec_ds.prec.where(prec_rmask == idx).where(prec_ds.prec >= 1).mean(dim=['lat2d','lon2d'], skipna=True)
        prec_regional.prec_mean.loc[{'region': reg}] = prec_wshed.values

    prec_regional.to_netcdf(f'/glade/work/tcorrie/ARdata/pmean/prec_mean.cmip6bc.{model}.{member}.nc')

access-cm2/r5i1p1f1
canesm5/r1i1p2f1
cesm2/r11i1p1f1
cnrm-esm2-1/r1i1p1f2
ec-earth3/r1i1p1f1
ec-earth3-veg/r1i1p1f1
fgoals-g3/r1i1p1f1
giss-e2-1-g/r1i1p1f2
miroc6/r1i1p1f1
mpi-esm1-2-hr/r3i1p1f1
mpi-esm1-2-hr/r7i1p1f1
mpi-esm1-2-lr/r7i1p1f1
noresm2-mm/r1i1p1f1
taiesm1/r1i1p1f1
ukesm1-0-ll/r2i1p1f2


In [14]:
# CESM2-LE ensemble

regions = gpd.read_file('/glade/u/home/tcorrie/watersheds_WUS.geojson', driver='GeoJSON')

for row in wusd3_cesm2le.index:
    model = wusd3_cesm2le.loc[row, 'Model']
    member = wusd3_cesm2le.loc[row, 'Member']
    print(f"{model}/{member}")

    file = f'/glade/derecho/scratch/tcorrie/WRFGCM/prec/prec.daily.cesm2-le.{member}.glob.nc'
    prec_ds = xa.open_dataset(file, use_cftime=True).load()
    prec_rmask = regionmask.mask_geopandas(regions, prec_ds.XLONG, prec_ds.XLAT)
    
    prec_regional = xa.Dataset(
        coords = dict(time=("time", prec_ds.day.values), region=('region', regions.huc4.values)),
        data_vars = dict(prec_mean=(['time', 'region'], np.empty((len(prec_ds.day.values), len(regions.huc4.values)))))
    )
    
    for idx, reg in enumerate(regions.huc4.values):
        prec_wshed = prec_ds.prec.where(prec_rmask == idx).where(prec_ds.prec >= 1).mean(dim=['lat2d','lon2d'], skipna=True)
        prec_regional.prec_mean.loc[{'region': reg}] = prec_wshed.values

    prec_regional.to_netcdf(f'/glade/work/tcorrie/ARdata/pmean/prec_mean.cesm2le.{model}.{member}.nc')

cesm2-le/1011
cesm2-le/1031
cesm2-le/1051
cesm2-le/1071
cesm2-le/1091
cesm2-le/1111
cesm2-le/1131
cesm2-le/1151
cesm2-le/1171
cesm2-le/1191


In [16]:
# WRF-ERA5

regions = gpd.read_file('/glade/u/home/tcorrie/watersheds_WUS.geojson', driver='GeoJSON')


model = wusd3_wrfera5.loc[6, 'Model']
member = wusd3_wrfera5.loc[6, 'Member']
print(f"{model}/{member}")

file = f'/glade/derecho/scratch/tcorrie/WRFGCM/prec/prec.daily.era5.glob.nc'
prec_ds = xa.open_dataset(file, use_cftime=True).load()
prec_rmask = regionmask.mask_geopandas(regions, prec_ds.XLONG, prec_ds.XLAT)

prec_regional = xa.Dataset(
    coords = dict(time=("time", prec_ds.day.values), region=('region', regions.huc4.values)),
    data_vars = dict(prec_mean=(['time', 'region'], np.empty((len(prec_ds.day.values), len(regions.huc4.values)))))
)

for idx, reg in enumerate(regions.huc4.values):
    prec_wshed = prec_ds.prec.where(prec_rmask == idx).where(prec_ds.prec >= 1).mean(dim=['lat2d','lon2d'], skipna=True)
    prec_regional.prec_mean.loc[{'region': reg}] = prec_wshed.values

prec_regional.to_netcdf(f'/glade/work/tcorrie/ARdata/pmean/prec_mean.wrfera5.{model}.{member}.nc')

era5/nan


In [3]:
# PRISM

regions = gpd.read_file('/glade/u/home/tcorrie/watersheds_WUS.geojson', driver='GeoJSON')


print(f"PRISM")

file = f'/glade/campaign/uwyo/wyom0200/tcorrie/PRISM/daily/prism_ppt_us_25m.nc'
prec_ds = xa.open_dataset(file, use_cftime=True).sel(lat=slice(31.5, 50), lon=slice(-130, -113)).load()
prec_rmask = regionmask.mask_geopandas(regions, prec_ds.lon, prec_ds.lat)

prec_regional = xa.Dataset(
    coords = dict(time=("time", prec_ds.time.values), region=('region', regions.huc4.values)),
    data_vars = dict(prec_mean=(['time', 'region'], np.empty((len(prec_ds.time.values), len(regions.huc4.values)))))
)

for idx, reg in enumerate(regions.huc4.values):
    prec_wshed = prec_ds.ppt.where(prec_rmask == idx).where(prec_ds.ppt >= 1).mean(dim=['lat','lon'], skipna=True)
    prec_regional.prec_mean.loc[{'region': reg}] = prec_wshed.values

prec_regional.to_netcdf(f'/glade/work/tcorrie/ARdata/pmean/prec_mean.prism.prism.nan.nc')

PRISM
